# IPO Market Gains

### Table of Contents

### Intro

When a company decides to go public, it embarks on a complex financial journey that transforms its ownership and market valuation. Through an initial public offering (IPO), the company works with investment banks to set an initial stock price that reflects its perceived value. However, the true test comes on the first day of public trading, when investors' collective judgment determines the stock's actual worth. This moment can result in a listing gain, where the stock price rises above the initial offering price, signify a neutral market response, or even experience a price decline, depending on investor confidence, market conditions, and the company's perceived potential.

In this project, we will build a deep learning model that predicts weather or not a business in the Indian Market will have positive listing gains. Our goal is to exemplify some of the very useful things pytorch and deep learning models can do for us and to demonstrate the different tools we can use to help our model be more accurate. We will be using a dataset is based off of [moneycontrol](https://www.moneycontrol.com/ipo/listed-ipos/?classic=true). It consists details from former IPOs in the Indian market. 

### Set Up

First things first, in the code below, we will import our libraries and bring in our data.

In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
%matplotlib inline

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix
import copy

In [2]:
df = pd.read_csv('Indian_IPO_Market_Data.csv')

### Intial Exploration

With our libraries and data now ready, we will do some exploring and make our first observations of the data in the code below.

In [3]:
df

,Date,IPOName,Issue_Size,Subscription_QIB,Subscription_HNI,Subscription_RII,Subscription_Total,Issue_Price,Listing_Gains_Percent
0,03/02/10,Infinite Comp,189.80,48.44,106.02,11.08,43.22,165,11.82
1,08/02/10,Jubilant Food,328.70,59.39,51.95,3.79,31.11,145,-84.21
2,15/02/10,Syncom Health,56.25,0.99,16.60,6.25,5.17,75,17.13
3,15/02/10,Vascon Engineer,199.80,1.12,3.65,0.62,1.22,165,-11.28
4,19/02/10,Thangamayil,0.00,0.52,1.52,2.26,1.12,75,-5.20
...,...,...,...,...,...,...,...,...,...
314,26/08/22,Syrma SGS,840.13,42.42,7.13,2.84,15.59,220,42.30
315,06/09/22,Dreamfolks Serv,562.10,27.48,14.18,24.19,23.25,326,41.92
316,15/09/22,TMB,792.00,0.51,1.77,3.44,1.39,525,-3.15
317,26/09/22,Harsha Engineer,755.00,113.82,40.36,12.44,47.19,330,47.24


**`df` – Observations:**

* There are **319** rows and **9** columns.
* We can observe a few categorical columns. 
* Additionally, we do not notice any empty or 'NaN' values. 

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 319 entries, 0 to 318
Data columns (total 9 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Date                   319 non-null    object 
 1   IPOName                319 non-null    object 
 2   Issue_Size             319 non-null    float64
 3   Subscription_QIB       319 non-null    float64
 4   Subscription_HNI       319 non-null    float64
 5   Subscription_RII       319 non-null    float64
 6   Subscription_Total     319 non-null    float64
 7   Issue_Price            319 non-null    int64  
 8   Listing_Gains_Percent  319 non-null    float64
dtypes: float64(6), int64(1), object(2)
memory usage: 22.6+ KB


**`df.info()` – Observations:**

* There are **2** object columns, **1** integer column, and **6** float columns.
* As we suspected, there are no missing values.

In [5]:
df.describe()

,Issue_Size,Subscription_QIB,Subscription_HNI,Subscription_RII,Subscription_Total,Issue_Price,Listing_Gains_Percent
count,319.000000,319.000000,319.000000,319.000000,319.000000,319.000000,319.000000
mean,1192.859969,25.684138,70.091379,8.561599,27.447147,375.128527,4.742696
std,2384.643786,40.716782,142.454416,14.508670,48.772203,353.897614,47.650946
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,-97.150000
25%,169.005000,1.150000,1.255000,1.275000,1.645000,119.000000,-11.555000
50%,496.250000,4.940000,5.070000,3.420000,4.930000,250.000000,1.810000
75%,1100.000000,34.635000,62.095000,8.605000,33.395000,536.000000,25.310000
max,21000.000000,215.450000,958.070000,119.440000,326.490000,2150.000000,270.400000


**`df.describe()` – Observations:**

* There do not appear to be any binary columns.

### Dataset Column Definitions<a id="def"></a>

1. **Date** – date when the IPO was listed
2. **IPOName** – name of the IPO
3. **Issue_Size** – size of the IPO issue, in INR Crores
4. **Subscription_QIB** – number of times the IPO was subscribed by QIB (Qualified Institutional Buyer) investors
5. **Subscription_HNI** – number of times the IPO was subscribed by HNI (High Networth Individual) investors
6. **Subscription_RII** – number of times the IPO was subscribed by RII (Retail Individual Investors)
7. **Subscription_Total** – total number of times the IPO was subscribed overall
8. **Issue_Price** – the price in INR at which the IPO was issued
9. **Listing_Gains_Percent** – percentage gain in the listing price over the issue price

### Further Exploration

Now that we've started to familarize ourselves with the dataset, we want to dive in a bit deeper. If we were in the position of an investor company, the main thing we care about when looking at an IPO is if it will be profitable or not. This is essentially what we will be trying to predict in our model later on. With this goal in mind, our next step of exploration will be to see how many instances in our dataset were profitable and how many were not. In the code below, we will filter the `Listing_Gains_Percent` column to form a new column that categorizes each instance as profitable or not. This new column will end up being our target label in our model.

In [6]:
df['Listing_Gains_Profit'] = (df['Listing_Gains_Percent'] > 0).astype(int)

In [7]:
profitable = df['Listing_Gains_Profit'].value_counts(normalize=True)[1] * 100
not_profitable = df['Listing_Gains_Profit'].value_counts(normalize=True)[0] * 100

print(f"Percentage of profitable IPOs: {profitable:.2f}%")
print(f"Percentage of unprofitable IPOs: {not_profitable:.2f}%")

Percentage of profitable IPOs: 54.55%
Percentage of unprofitable IPOs: 45.45%
